# 1. Load the data and look at it

Read data/interim/parsed.jsonl. This is the full table extracted from the XML. It has not been cleaned, and it has not been split.

The model later reads **text** and predicts two multi-label fields: 
rechtsgebieden and procedures. Year and area are only here to check the sample.

In [1]:
from pathlib import Path
import json
import pandas as pd

PARSED = Path('../data/interim/parsed.jsonl')
print(PARSED.resolve())

C:\Users\nlwen\Desktop\hbo-corp-justid\2026\data\interim\parsed.jsonl


Look at the first judgment first, to check the fields. The full table is about 1.2 GB; the next cell will take a minute or two to load.

In [2]:
with PARSED.open(encoding='utf-8') as f:
    first = json.loads(f.readline())

print('fields:', list(first))
print('ecli:', first['ecli'])
print('court:', first['court'])
print('date:', first['date'])
print('year / area / pool:', first['year'], first['area'], first['pool'])
print('rechtsgebieden:', first['rechtsgebieden'])
print('rechtsgebieden_raw:', first['rechtsgebieden_raw'])
print('procedures:', first['procedures'])
print('text_length:', first['text_length'], 'source:', first['text_source'])
print('text start:')
print(first['text'][:400])

fields: ['ecli', 'court', 'date', 'text', 'text_length', 'text_source', 'rechtsgebieden', 'rechtsgebieden_raw', 'procedures', 'pool', 'year', 'area']
ecli: ECLI:NL:CBB:2006:AV0094
court: College van Beroep voor het bedrijfsleven
date: 2006-01-11
year / area / pool: 2006 Bestuursrecht balanced
rechtsgebieden: ['Bestuursrecht']
rechtsgebieden_raw: ['Bestuursrecht']
procedures: ['Eerste aanleg - meervoudig', 'Eerste en enige aanleg']
text_length: 9143 source: uitspraak
text start:
College van Beroep voor het bedrijfsleven
AWB 05/71				11 januari 2006
10500 Superheffing
Uitspraak in de zaak van:
A, te X, appellant,
tegen
het Productschap Zuivel, verweerder,
gemachtigden: mr. G.W.P.A. van Schijndel en L.J. Koers, beiden werkzaam bij verweerder.
1.	De procedure
Appellant heeft bij brief van 25 januari 2005, bij het College binnengekomen op 28 januari 2005, beroep ingesteld teg


In [3]:
df = pd.read_json(PARSED, lines=True)
print('rows:', len(df))
print('columns:', list(df.columns))
df.dtypes

rows: 65480
columns: ['ecli', 'court', 'date', 'text', 'text_length', 'text_source', 'rechtsgebieden', 'rechtsgebieden_raw', 'procedures', 'pool', 'year', 'area']


ecli                          object
court                         object
date                  datetime64[ns]
text                          object
text_length                    int64
text_source                   object
rechtsgebieden                object
rechtsgebieden_raw            object
procedures                    object
pool                          object
year                           int64
area                          object
dtype: object

In [4]:
preview = df.head(5).copy()
preview['text'] = preview['text'].str.slice(0, 120) + '…'
preview[
    ['ecli', 'court', 'date', 'year', 'area', 'pool',
     'rechtsgebieden', 'procedures', 'text_length', 'text']
]

,ecli,court,date,year,area,pool,rechtsgebieden,procedures,text_length,text
0,ECLI:NL:CBB:2006:AV0094,College van Beroep voor het bedrijfsleven,2006-01-11,2006,Bestuursrecht,balanced,[Bestuursrecht],"[Eerste aanleg - meervoudig, Eerste en enige a...",9143,College van Beroep voor het bedrijfsleven\nAWB...
1,ECLI:NL:CBB:2006:AV2086,College van Beroep voor het bedrijfsleven,2006-02-03,2006,Bestuursrecht,balanced,[Bestuursrecht],"[Eerste aanleg - meervoudig, Proceskostenveroo...",29526,College van Beroep voor het bedrijfsleven\nAWB...
2,ECLI:NL:CBB:2006:AV2126,College van Beroep voor het bedrijfsleven,2006-02-10,2006,Bestuursrecht,balanced,[Bestuursrecht],"[Eerste aanleg - meervoudig, Eerste en enige a...",10411,College van Beroep voor het bedrijfsleven\nAWB...
3,ECLI:NL:CBB:2006:AV2686,College van Beroep voor het bedrijfsleven,2006-01-31,2006,Bestuursrecht,balanced,[Bestuursrecht],"[Eerste aanleg - meervoudig, Eerste en enige a...",14566,College van Beroep voor het bedrijfsleven\nAWB...
4,ECLI:NL:CBB:2006:AV2917,College van Beroep voor het bedrijfsleven,2006-02-15,2006,Bestuursrecht,balanced,[Bestuursrecht],"[Eerste aanleg - enkelvoudig, Proceskostenvero...",20403,College van Beroep voor het bedrijfsleven\n(ze...


## What kind of text

	ext_source says where the body came from. Almost everything should be uitspraak (the court ruling). A few files are conclusie (an opinion). inhoudsindicatie is only a short summary.

In [5]:
print(df['text_source'].value_counts(dropna=False))
print('empty text:', (df['text_length'] == 0).sum())

text_source
uitspraak           62479
conclusie            2999
                        1
inhoudsindicatie        1
Name: count, dtype: int64
empty text: 1


How long are the texts? Very short ones are often stubs. We do not drop them in this notebook.

In [6]:
print(df['text_length'].describe().round(0))
print()
print('shorter than 200 chars:', (df['text_length'] < 200).sum())
print('shorter than 500 chars:', (df['text_length'] < 500).sum())
print('shorter than 1000 chars:', (df['text_length'] < 1000).sum())

count      65480.0
mean       17877.0
std        21565.0
min            0.0
25%         7315.0
50%        12424.0
75%        21452.0
max      1055682.0
Name: text_length, dtype: float64

shorter than 200 chars: 23
shorter than 500 chars: 164
shorter than 1000 chars: 196


## The two label fields

One row can have several labels. Count how many sit on each document, then how often each label appears.

In [7]:
df['n_rg'] = df['rechtsgebieden'].map(len)
df['n_pr'] = df['procedures'].map(len)

print('rechtsgebieden per document')
print(df['n_rg'].value_counts().sort_index())
print('no rechtsgebieden:', (df['n_rg'] == 0).sum())
print()
print('procedures per document')
print(df['n_pr'].value_counts().sort_index())
print('no procedures:', (df['n_pr'] == 0).sum())

rechtsgebieden per document
n_rg
1    40730
2    23755
3      950
4       41
5        4
Name: count, dtype: int64
no rechtsgebieden: 0

procedures per document


n_pr
0     3011
1    56826
2     5094
3      482
4       55
5       10
6        2
Name: count, dtype: int64
no procedures: 3011


In [8]:
rg = df.explode('rechtsgebieden')['rechtsgebieden'].value_counts()
print('rechtsgebieden labels:', len(rg))
rg

rechtsgebieden labels: 31


rechtsgebieden
Bestuursrecht                   22793
Civiel recht                    21731
Strafrecht                      21386
Socialezekerheidsrecht           5540
Belastingrecht                   4084
Personen- en familierecht        3991
Vreemdelingenrecht               3470
Verbintenissenrecht              1643
Omgevingsrecht                   1114
Ambtenarenrecht                   738
Insolventierecht                  703
Arbeidsrecht                      689
Internationaal publiekrecht       535
Europees strafrecht               452
Bestuursstrafrecht                446
Bestuursprocesrecht               354
Materieel strafrecht              318
Burgerlijk procesrecht            310
Strafprocesrecht                  244
Ondernemingsrecht                 236
Internationaal strafrecht         128
Intellectueel-eigendomsrecht      101
Goederenrecht                      61
Aanbestedingsrecht                 49
Internationaal privaatrecht        47
Penitentiair strafrecht            

In [9]:
pr = df.explode('procedures')['procedures'].value_counts()
print('procedure labels:', len(pr))
pr

procedure labels: 43


procedures
Hoger beroep                            20370
Eerste aanleg - enkelvoudig             14451
Eerste aanleg - meervoudig              14256
Cassatie                                 3630
Op tegenspraak                           2425
Kort geding                              2255
Beschikking                              2022
Voorlopige voorziening                   1740
Bodemzaak                                1227
Eerste en enige aanleg                    710
Rekestprocedure                           669
Artikel 81 RO-zaken                       588
Wraking                                   575
Raadkamer                                 525
Hoger beroep kort geding                  437
Voorlopige voorziening+bodemzaak          408
Tussenuitspraak                           356
Proceskostenveroordeling                  296
Mondelinge uitspraak                      281
Herziening                                205
Proces-verbaal                            190
Verzet                 

Labels that appear only a handful of times are hard to learn. Nothing is dropped here.

In [10]:
def rare_counts(counts):
    return {
        '< 20': int((counts < 20).sum()),
        '< 50': int((counts < 50).sum()),
        '< 100': int((counts < 100).sum()),
        'total labels': int(len(counts)),
    }

print('rechtsgebieden', rare_counts(rg))
print('procedures', rare_counts(pr))

rechtsgebieden {'< 20': 3, '< 50': 8, '< 100': 9, 'total labels': 31}
procedures {'< 20': 13, '< 50': 14, '< 100': 17, 'total labels': 43}


## Sample check

year, rea, and pool were used to draw the sample. They are not model inputs. alanced should be spread across years and the four top-level areas. 
atural_test is the leftover real-world mix.

In [11]:
print(df['pool'].value_counts())
print()
print('area')
print(df['area'].value_counts())
print()
print('year x area (balanced only)')
balanced = df[df['pool'] == 'balanced']
print(pd.crosstab(balanced['year'], balanced['area']))

pool
balanced        60480
natural_test     5000
Name: count, dtype: int64

area
area
Bestuursrecht                  22406
Civiel recht                   21549
Strafrecht                     20994
Internationaal publiekrecht      531
Name: count, dtype: int64

year x area (balanced only)


area  Bestuursrecht  Civiel recht  Internationaal publiekrecht  Strafrecht
year                                                                      
2006           1000           998                            0         999
2007           1000          1000                            0        1000
2008           1000           999                            0        1000
2009           1000           990                            0         999
2010           1000          1000                            0        1000
2011           1000          1000                            0        1000
2012           1000          1000                            0        1000
2013           1000          1000                            2         999
2014           1000          1000                           33         997
2015           1000          1000                           41        1000
2016           1000          1000                           17        1000
2017           1000      